# NewsAgent Scheduler Demo: BOCHA Web Search

This notebook demonstrates:
1. **Environment override** - Test with custom settings without modifying `.env`
2. **Real API call** - First execution with real BOCHA API
3. **Fake response caching** - Second execution using cached response
4. **Database retrieval** - Query results from database

Based on: `legacy/BOCHA_news_acquisition.ipynb`

## Step 1: Import Required Modules

In [1]:
import json
from datetime import datetime
import os
from pathlib import Path
import sys

# Add parent directory to path so imports work from scripts/
parent_dir = str(Path.cwd().parent.resolve())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# NewsAgent modules
from src.scheduler.scheduler_settings import SchedulerSettings
from src.debug_config import DebugConfig
from src.dataclasses import QueryRequest
from src.agents.bocha import BochaAgent
from src.database import SQLite3Backend, DatabaseManager
from src.utils import fake_response_manager

print("✓ All modules imported successfully")

✓ All modules imported successfully


## Step 2: Initialize Settings with Environment Overrides

Using the **single entry point pattern** with overrides for testing:

In [2]:
print("=" * 70)
print("INITIALIZING SCHEDULER SETTINGS WITH OVERRIDES")
print("=" * 70)
print()

# Initialize with environment overrides
# - Loads from .env file
# - Applies overrides for testing
settings = SchedulerSettings.initialize(
    env_file=".env",                              # Use standard .env
    database_path="data/newsagent.db",  # Override database
    log_level="DEBUG"                             # Increase verbosity
)

print(f"✓ Settings initialized")
print(f"  Database: {settings.env_vars.database_path}")
print(f"  Log Level: {settings.env_vars.log_level}")
print(f"  BOCHA API Key: {'***' if settings.env_vars.bocha_api_key else 'NOT SET'}")
print()

# Check agent availability
ready_agents = settings.get_ready_agents()
print(f"Ready agents: {list(ready_agents.keys())}")

if "BOCHA" not in ready_agents:
    print("\n❌ ERROR: BOCHA_API_KEY not set in .env")
    print("Please add BOCHA_API_KEY to your .env file")
    raise ValueError("BOCHA API key not configured")

print("\n✓ BOCHA agent is ready")

INITIALIZING SCHEDULER SETTINGS WITH OVERRIDES

✓ Settings initialized
  Database: data/newsagent.db
  Log Level: DEBUG
  BOCHA API Key: ***

Ready agents: ['XUNFEI', 'BOCHA', 'HUNYUAN', 'QIANFAN', 'META', 'TWITTER']

✓ BOCHA agent is ready


## Step 3: Configure Debug Settings for First Run (Real API)

First execution will call the real API and cache the response:

In [3]:
print("=" * 70)
print("CONFIGURING DEBUG MODE - FIRST RUN (REAL API + CACHE)")
print("=" * 70)
print()

# Configure debug flags for real API call with caching
DebugConfig.DEBUG = True
DebugConfig.fake_response_enabled = True   # Enable caching system
DebugConfig.fake_response_update = True    # Save responses to cache
DebugConfig.fake_response_interact = False # No interactive prompts

print("Debug Configuration:")
print(f"  DEBUG: {DebugConfig.DEBUG}")
print(f"  fake_response_enabled: {DebugConfig.fake_response_enabled}")
print(f"  fake_response_update: {DebugConfig.fake_response_update}")
print(f"  fake_response_interact: {DebugConfig.fake_response_interact}")
print()
print("Expected behavior: Call real API and cache the response")

CONFIGURING DEBUG MODE - FIRST RUN (REAL API + CACHE)

Debug Configuration:
  DEBUG: True
  fake_response_enabled: True
  fake_response_update: True
  fake_response_interact: False

Expected behavior: Call real API and cache the response


## Step 4: Setup Database Connection

In [4]:
print("=" * 70)
print("SETTING UP DATABASE")
print("=" * 70)
print()

# Create database backend and manager
db_backend = SQLite3Backend(db_path=settings.env_vars.database_path)
db_manager = DatabaseManager(db_backend)
db_manager.connect()

print(f"✓ Database connected: {settings.env_vars.database_path}")
print()

# Show initial stats
db_manager.print_stats()

SETTING UP DATABASE

✓ Database connected: data/newsagent.db


DATABASE STATISTICS
Total Queries: 3
Total Responses: 4
Total Items: 0
Unique Sources: 0

Responses by Agent:
  BOCHA: 4



## Step 5: Initialize BOCHA Agent

In [5]:
print("=" * 70)
print("INITIALIZING BOCHA AGENT")
print("=" * 70)
print()

# Get BOCHA config from ready agents
bocha_config = ready_agents["BOCHA"]
bocha_agent = BochaAgent(bocha_config)

print(f"✓ BOCHA agent initialized")
print(f"  Agent name: {bocha_config.agent_name}")
print(f"  Endpoint: {bocha_config.api_endpoint}")
print(f"  Supports time filter: {bocha_config.supports_time_filter}")
print(f"  Supports AI summary: {bocha_config.supports_ai_summary}")

INITIALIZING BOCHA AGENT

✓ BOCHA agent initialized
  Agent name: BOCHA
  Endpoint: https://api.bochaai.com/v1/web-search
  Supports time filter: True
  Supports AI summary: True


## Step 6: Define Search Query

Based on the legacy notebook's search parameters:

In [6]:
print("=" * 70)
print("CREATING SEARCH QUERY")
print("=" * 70)
print()

# Define technology fields and focus topics (from legacy notebook)
TECH_FIELDS = [
    "自动驾驶",
    "具身智能",
    "大模型",
    # "人工智能"
]

FOCUS_TOPICS = [
    "特斯拉FSD",
    "人形机器人",
    "AI大模型突破",
    "行业重要动态"
]

# Create QueryRequest
query = QueryRequest(
    query_fields=TECH_FIELDS,
    query_topics=FOCUS_TOPICS,
    source_agents=["BOCHA"],
    days_back=7,                  # Last week
    max_results=8,                # 8 results (from legacy notebook)
    include_ai_summary=True,      # Enable AI summary
    include_raw_response=True,    # Store raw API response in JSON format
    language="zh"                 # Chinese
)

print(f"Query created:")
print(f"  Query ID: {query.query_id}")
print(f"  Fields: {', '.join(query.query_fields)}")
print(f"  Topics: {', '.join(query.query_topics)}")
print(f"  Time range: {query.days_back} days")
print(f"  Max results: {query.max_results}")
print(f"  AI summary: {query.include_ai_summary}")
print(f"  Include raw response: {query.include_raw_response}")

CREATING SEARCH QUERY

Query created:
  Query ID: 282180a6-f6f1-463e-a2b6-7811b18e6b0d
  Fields: 自动驾驶, 具身智能, 大模型
  Topics: 特斯拉FSD, 人形机器人, AI大模型突破, 行业重要动态
  Time range: 7 days
  Max results: 8
  AI summary: True
  Include raw response: True


## Step 7: Execute Query - First Time (Real API Call)

This will call the real BOCHA API and cache the response:

In [7]:
print("\n" + "=" * 70)
print("EXECUTION #1 - REAL API CALL (WITH CACHING)")
print("=" * 70)
print()

print("Calling BOCHA API...")
print("(This will call the real API and save response to cache)")
print()

# Save query to database first
db_manager.save_query(query)
print(f"✓ Query saved to database (ID: {query.query_id})")
print()

# Execute query (decorated with @fake_response_handler)
import time
start_time = time.time()

response_1 = bocha_agent.submit_and_parse(query)

execution_time_1 = time.time() - start_time

print()
print("=" * 70)
print("FIRST EXECUTION RESULTS")
print("=" * 70)

if response_1.success:
    print(f"\n✓ Success!")
    print(f"  Items retrieved: {len(response_1.items)}")
    print(f"  Execution time: {execution_time_1:.2f}s")
    print(f"  Status: {response_1.status}")
    
    # Save response to database
    db_manager.save_response(response_1)
    print(f"\n✓ Response saved to database (ID: {response_1.response_id})")
    
    # Save items to database
    if response_1.items:
        db_manager.save_items_batch(response_1.items)
        print(f"✓ {len(response_1.items)} items saved to database")
    
    # Show first 2 results
    print("\n" + "-" * 70)
    print("SAMPLE RESULTS (First 2 items):")
    print("-" * 70)
    
    for i, item in enumerate(response_1.items[:2], 1):
        print(f"\n[{i}] {item.title}")
        print(f"    Source: {item.source_name}")
        print(f"    URL: {item.source_url}")
        print(f"    Published: {item.timestamp.strftime('%Y-%m-%d %H:%M')}")
        print(f"    Content: {item.content[:150]}...")
        
        # Show AI summary if available
        if 'summary' in item.metadata and item.metadata['summary']:
            summary = item.metadata['summary']
            print(f"    💡 AI Summary: {summary[:200]}...")
else:
    print(f"\n❌ Query failed: {response_1.error_message}")


EXECUTION #1 - REAL API CALL (WITH CACHING)

Calling BOCHA API...
(This will call the real API and save response to cache)

✓ Query saved to database (ID: 282180a6-f6f1-463e-a2b6-7811b18e6b0d)

[23:01:39] [INFO] [src.decorators.response_handler] Using cached response: BOCHA/4711c790dd50380aca7ec48b34bb23e2

FIRST EXECUTION RESULTS

✓ Success!
  Items retrieved: 0
  Execution time: 0.00s
  Status: completed

✓ Response saved to database (ID: df56b1e2-6c0a-4981-b7fa-0dc28323c35f)

----------------------------------------------------------------------
SAMPLE RESULTS (First 2 items):
----------------------------------------------------------------------


In [8]:
response_1.raw_response

{'url': 'https://api.bochaai.com/v1/web-search',
 'method': 'POST',
 'description': 'call_api',
 'timestamp_created': '2025-11-18T22:49:05.876221',
 'timestamp_updated': '2025-11-18T22:49:05.876232',
 'request_body': {'query_fields': [],
  'query_topics': [],
  'source_agents': [],
  'days_back': 7,
  'max_results': 10,
  'language': 'en'},
 'response_body': {'code': 200,
  'log_id': 'e9d9c19780008272',
  'msg': None,
  'data': {'_type': 'SearchResponse',
   'queryContext': {'originalQuery': '自动驾驶 具身智能 大模型 人工智能 特斯拉FSD 人形机器人 AI大模型突破 行业重要动态'},
   'webPages': {'webSearchUrl': 'https://bochaai.com/search?q=自动驾驶 具身智能 大模型 人工智能 特斯拉FSD 人形机器人 AI大模型突破 行业重要动态',
    'totalEstimatedMatches': 2413070,
    'value': [{'id': 'https://api.bochaai.com/v1/#WebPages.0',
      'name': '财经(2025年第23期) - 财经 - 微信读书',
      'url': 'https://weread.qq.com/web/reader/66932870813abab85g016860',
      'displayUrl': 'https://weread.qq.com/web/reader/66932870813abab85g016860',
      'snippet': '简介大量资本涌入具身智能,尤其是人形机器人赛道,

## Step 8: Check Cached Response

Verify that the response was cached:

In [10]:
cached_responses

[{'md5_hash': '4711c790dd50380aca7ec48b34bb23e2',
  'agent': 'BOCHA',
  'url': 'https://api.bochaai.com/v1/web-search',
  'method': 'POST',
  'description': 'call_api',
  'usage_count': 0,
  'created': '2025-11-18T22:49:05.877243',
  'last_used': None,
  'notes': 'Auto-cached from API call'}]

In [9]:
print("\n" + "=" * 70)
print("CHECKING CACHE STATUS")
print("=" * 70)
print()

# List cached responses for BOCHA
cached_responses = fake_response_manager.list_responses("BOCHA")
print(f"Total cached BOCHA responses: {len(cached_responses)}")
print()

if cached_responses:
    print("Cached responses:")
    for i, resp in enumerate(cached_responses, 1):
        print(f"  {i}. {resp['description']}")
        print(f"     File: {resp['file_path']}")
        print(f"     Size: {resp['size_bytes']:,} bytes")
        print()

# Get statistics
stats = fake_response_manager.get_statistics("BOCHA")
print("Cache statistics:")
print(f"  Total responses: {stats['total_responses']}")
print(f"  Total size: {stats['total_size_bytes']:,} bytes")
print(f"  Cache directory: {stats['cache_directory']}")


CHECKING CACHE STATUS

Total cached BOCHA responses: 1

Cached responses:
  1. call_api


KeyError: 'file_path'

## Step 9: Execute Query - Second Time (From Cache)

This will retrieve the response from cache (instant):

In [ ]:
print("\n" + "=" * 70)
print("EXECUTION #2 - FROM CACHE (INSTANT)")
print("=" * 70)
print()

# Debug config is already set (fake_response_enabled=True)
print("Executing same query again...")
print("(This should load from cache instantly)")
print()

# Execute query again
start_time = time.time()

response_2 = bocha_agent.submit_and_parse(query)

execution_time_2 = time.time() - start_time

print()
print("=" * 70)
print("SECOND EXECUTION RESULTS")
print("=" * 70)

if response_2.success:
    print(f"\n✓ Success!")
    print(f"  Items retrieved: {len(response_2.items)}")
    print(f"  Execution time: {execution_time_2:.4f}s ⚡ (from cache!)")
    print(f"  Status: {response_2.status}")
    
    # Compare execution times
    print("\n" + "-" * 70)
    print("PERFORMANCE COMPARISON")
    print("-" * 70)
    print(f"  First run (real API):  {execution_time_1:.2f}s")
    print(f"  Second run (cache):    {execution_time_2:.4f}s")
    if execution_time_1 > 0 and execution_time_2 > 0:
        speedup = execution_time_1 / execution_time_2
        print(f"  Speedup: {speedup:.0f}x faster! 🚀")
else:
    print(f"\n❌ Query failed: {response_2.error_message}")

In [ ]:
print("\n" + "=" * 70)
print("INSPECTING CACHED RAW RESPONSE")
print("=" * 70)
print()

# Check if raw response was captured
if response_1.success and response_1.raw_response:
    raw_resp = response_1.raw_response
    
    print("✓ Raw API Response captured and cached!")
    print()
    print("Raw Response Structure:")
    print(f"  Keys: {list(raw_resp.keys())}")
    print()
    
    # Show webPages structure
    if 'webPages' in raw_resp:
        web_pages = raw_resp['webPages']
        print("webPages:")
        print(f"  totalEstimatedMatches: {web_pages.get('totalEstimatedMatches', 'N/A')}")
        print(f"  Number of results: {len(web_pages.get('value', []))}")
        print()
        
        # Show first result structure
        if web_pages.get('value'):
            first_result = web_pages['value'][0]
            print("First Result Structure:")
            for key in first_result.keys():
                value = first_result[key]
                if isinstance(value, str):
                    preview = value[:80] + "..." if len(value) > 80 else value
                    print(f"  {key}: {preview}")
                else:
                    print(f"  {key}: {type(value).__name__}")
    print()
    
    # Show cached file info
    cached_file = fake_response_manager.list_responses("BOCHA")
    if cached_file:
        print("Cached Response File:")
        cache_info = cached_file[0]
        print(f"  Agent: {cache_info.get('agent', 'N/A')}")
        print(f"  Hash: {cache_info.get('md5_hash', 'N/A')}")
        print(f"  Created: {cache_info.get('created', 'N/A')}")
        print(f"  Location: data/fake_response/bocha/{cache_info.get('md5_hash', 'N/A')}.json")
        print()
        
        # Show JSON file preview
        import json
        from pathlib import Path
        cache_dir = Path("data/fake_response/bocha")
        if cache_dir.exists():
            json_files = list(cache_dir.glob("*.json"))
            if json_files and not any(".metadata" in f.name for f in json_files):
                cache_file = [f for f in json_files if ".metadata" not in f.name][0]
                with open(cache_file, 'r', encoding='utf-8') as f:
                    cached_data = json.load(f)
                
                print(f"File size: {cache_file.stat().st_size:,} bytes")
                print()
                print("Cached File Structure:")
                print(f"  url: {cached_data.get('url', 'N/A')}")
                print(f"  method: {cached_data.get('method', 'N/A')}")
                print(f"  description: {cached_data.get('description', 'N/A')}")
                print(f"  timestamp_created: {cached_data.get('timestamp_created', 'N/A')}")
                print()
                
                if 'response_body' in cached_data:
                    resp_body = cached_data['response_body']
                    print("response_body keys:")
                    for key in resp_body.keys():
                        if key == 'items':
                            print(f"  {key}: List[{len(resp_body[key])} items]")
                        elif key == 'raw_response':
                            if resp_body[key]:
                                print(f"  {key}: {type(resp_body[key]).__name__} with keys: {list(resp_body[key].keys())}")
                            else:
                                print(f"  {key}: None")
                        else:
                            print(f"  {key}: {resp_body[key]}")
else:
    print("❌ Raw response not captured")
    print("   Make sure include_raw_response=True in QueryRequest")

## Step 9a: Inspect Cached Raw Response

View the original API response that was stored:

## Step 10: Retrieve Results from Database

Query the database to see what was saved:

In [ ]:
print("\n" + "=" * 70)
print("RETRIEVING FROM DATABASE")
print("=" * 70)
print()

# 1. Get queries
print("1. Recent Queries:")
recent_queries = db_manager.list_queries(limit=5)
print(f"   Total: {len(recent_queries)}")
if recent_queries:
    for q in recent_queries[:3]:
        print(f"   - {q.query_id[:8]}... | Fields: {', '.join(q.query_fields[:2])}...")
print()

# 2. Get responses
print("2. Recent Responses:")
recent_responses = db_manager.list_responses(agent_name="BOCHA", limit=5)
print(f"   Total BOCHA responses: {len(recent_responses)}")
if recent_responses:
    for r in recent_responses[:3]:
        print(f"   - {r.response_id[:8]}... | Agent: {r.agent_name} | Items: {len(r.items)} | Status: {r.status}")
print()

# 3. Get items
print("3. Recent Items:")
recent_items = db_manager.get_recent_items(days=7, limit=10)
print(f"   Total items (last 7 days): {len(recent_items)}")
if recent_items:
    for item in recent_items[:3]:
        print(f"   - {item.title[:50]}...")
        print(f"     Source: {item.source_name} | {item.timestamp.strftime('%Y-%m-%d')}")
print()

# 4. Search by source
print("4. Search Items by Source (BOCHA):")
bocha_items = db_manager.search_items(source_type="BOCHA", limit=5)
print(f"   Found {len(bocha_items)} items from BOCHA")
print()

# 5. Database statistics
print("5. Database Statistics:")
db_manager.print_stats()

## Step 11: Analyze Results

Statistical analysis of the retrieved data:

In [ ]:
print("\n" + "=" * 70)
print("RESULTS ANALYSIS")
print("=" * 70)
print()

if response_1.success and response_1.items:
    items = response_1.items
    
    # Source distribution
    print("1. Source Distribution:")
    source_count = {}
    for item in items:
        source = item.source_name or "Unknown"
        source_count[source] = source_count.get(source, 0) + 1
    
    for source, count in sorted(source_count.items(), key=lambda x: x[1], reverse=True):
        print(f"   {source}: {count} articles")
    print()
    
    # Date distribution
    print("2. Publication Date Distribution:")
    date_count = {}
    for item in items:
        date_str = item.timestamp.strftime('%Y-%m-%d')
        date_count[date_str] = date_count.get(date_str, 0) + 1
    
    for date, count in sorted(date_count.items(), reverse=True):
        print(f"   {date}: {count} articles")
    print()
    
    # AI Summary coverage
    print("3. AI Summary Coverage:")
    with_summary = sum(1 for item in items if item.metadata.get('summary'))
    percentage = (with_summary * 100 // len(items)) if items else 0
    print(f"   Articles with AI summary: {with_summary}/{len(items)} ({percentage}%)")
    print()
    
    # Average content length
    print("4. Content Statistics:")
    avg_content_len = sum(len(item.content) for item in items) // len(items) if items else 0
    print(f"   Average content length: {avg_content_len} characters")
    print(f"   Total items: {len(items)}")
else:
    print("No items to analyze")

## Step 12: Cleanup and Summary

In [ ]:
# Export first execution results
if response_1.success and response_1.items:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"data/demo_bocha_results_{timestamp}.json"
    
    # Prepare export data
    export_data = {
        "query": {
            "query_id": query.query_id,
            "fields": query.query_fields,
            "topics": query.query_topics,
            "days_back": query.days_back,
            "max_results": query.max_results,
            "include_raw_response": query.include_raw_response
        },
        "response": {
            "response_id": response_1.response_id,
            "agent_name": response_1.agent_name,
            "success": response_1.success,
            "status": response_1.status,
            "execution_time_ms": response_1.execution_time_ms,
            "total_items": len(response_1.items),
            "total_estimated": response_1.total_estimated
        },
        "items": [
            {
                "title": item.title,
                "source_name": item.source_name,
                "source_url": item.source_url,
                "timestamp": item.timestamp.isoformat(),
                "content": item.content,
                "summary": item.metadata.get('summary', '')
            }
            for item in response_1.items
        ]
    }
    
    # Include raw API response if available
    if response_1.raw_response:
        export_data["raw_api_response"] = response_1.raw_response
        print("\n✓ Raw API response included in export")
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Results exported to: {output_file}")
    import os
    file_size = os.path.getsize(output_file)
    print(f"  File size: {file_size:,} bytes")
else:
    print("No results to export")

## Optional: Export Results to JSON

Save the results for further analysis:

In [ ]:
# Export first execution results
if response_1.success and response_1.items:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"data/demo_bocha_results_{timestamp}.json"
    
    # Prepare export data
    export_data = {
        "query": {
            "query_id": query.query_id,
            "fields": query.query_fields,
            "topics": query.query_topics,
            "days_back": query.days_back,
            "max_results": query.max_results
        },
        "response": {
            "response_id": response_1.response_id,
            "agent_name": response_1.agent_name,
            "success": response_1.success,
            "status": response_1.status,
            "execution_time_ms": response_1.execution_time_ms,
            "total_items": len(response_1.items)
        },
        "items": [
            {
                "title": item.title,
                "source_name": item.source_name,
                "source_url": item.source_url,
                "timestamp": item.timestamp.isoformat(),
                "content": item.content,
                "summary": item.metadata.get('summary', '')
            }
            for item in response_1.items
        ]
    }
    
    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Results exported to: {output_file}")
    import os
    file_size = os.path.getsize(output_file)
    print(f"  File size: {file_size:,} bytes")
else:
    print("No results to export")

## Summary

This notebook demonstrated the complete NewsAgent workflow:

### ✅ Accomplished

1. **Environment Override** - Used `SchedulerSettings.initialize()` with custom database path
2. **Real API Call** - First execution called BOCHA API and cached response
3. **Cached Response** - Second execution loaded from cache (instant)
4. **Database Storage** - Saved queries, responses, and items
5. **Database Retrieval** - Retrieved and analyzed results

### 🎯 Key Patterns Used

- **Single Entry Point**: `SchedulerSettings.initialize()` with `**overrides`
- **Fake Response Caching**: `@fake_response_handler` decorator
- **Database Persistence**: SQLite3Backend with DatabaseManager
- **Type-Safe Models**: QueryRequest, QueryResponse, SearchItem

### 📚 Related Documentation

- Environment Override Guide: `docs/ENVIRONMENT_OVERRIDE_QUICK_REFERENCE.md`
- Fake Response Manual: `docs/FAKE_RESPONSE_MANUAL.md`
- Configuration Guide: `docs/CONFIGURATION_GUIDE.md`
- Original Implementation: `legacy/BOCHA_news_acquisition.ipynb`